# Pathwiser 02: training-free baseline results

This notebook **computes nothing**; it reads and interprets the results written by `pathwiser baselines run` (`reports/baselines/v1/*.json`).
The only exception is the threshold curve in §6, which recomputes the scores of the best baseline on **validation** only.

**Protocol** (`docs/evaluation-protocol.md`)
* The decision threshold τ is tuned on **validation main examples** (objective: decision accuracy) and applied unchanged to test.
  Validation numbers are therefore optimistic, **test numbers are the honest ones**.
* SGD-X paraphrase twins are not mixed into the totals; they are reported separately.
* `random` is the chance reference; `lexical*` and `tfidf*` answer "how much is solvable by word overlap alone".
* Scores are not probabilities; calibration is not reported.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)


def find_root(start: Path | None = None) -> Path:
    start = start or Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "pathwiser").is_dir():
            return p
    raise FileNotFoundError("repo root not found")


ROOT = find_root()
sys.path.insert(0, str(ROOT))
REPORTS = ROOT / "reports" / "baselines" / "v1"

ORDER = ["random", "lexical", "lexical_ctx", "tfidf", "tfidf_ctx", "st_message", "st_query"]
RES = {n: json.loads((REPORTS / f"{n}.json").read_text(encoding="utf-8")) for n in ORDER if (REPORTS / f"{n}.json").exists()}
HAS_TEST = all("test" in r["splits"] for r in RES.values())
SHOWN = "test" if HAS_TEST else "val"

# Fixed colour per baseline, identical in every chart
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6", "#008300", "#e34948"]
COLOR = {"random": "#9a9893", "lexical": PAL[0], "lexical_ctx": PAL[1], "tfidf": PAL[2], "tfidf_ctx": PAL[3],
         "st_message": PAL[5], "st_query": PAL[4]}
INK, INK2 = "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.facecolor": "#fcfcfb", "figure.facecolor": "#fcfcfb", "legend.frameon": False,
})
d = next(iter(RES.values()))["data"]
print("baselines:", list(RES))
print("shown split:", SHOWN, "| data version:", d["version"], "| test sha:", d["test_sha256"][:12])

## 1. Headline results

In [ ]:
HEAD = [("top1", "Top-1"), ("top3", "Top-3"), ("mrr", "MRR"), ("decision_accuracy", "Decision acc."),
        ("no_match_precision", "no_match P"), ("no_match_recall", "no_match R"), ("false_routing_rate", "False routing")]


def headline(split: str) -> pd.DataFrame:
    rows = {n: {h: r["splits"][split]["overall"][k] for k, h in HEAD} | {"ms/example": r["latency_ms_per_example"][split]}
            for n, r in RES.items() if split in r["splits"]}
    return pd.DataFrame(rows).T


print("== Validation (τ tuned on this split, optimistic) ==")
display(headline("val").round(3))
if HAS_TEST:
    print("== Test (τ from validation), honest numbers ==")
    display(headline("test").round(3))

h = headline(SHOWN)
fig, ax = plt.subplots(figsize=(8.5, 3.6))
y = np.arange(len(h)); w = 0.38
for off, col, lab, shade in [(-w / 2, "Top-1", "Top-1", 1.0), (w / 2, "Decision acc.", "Decision accuracy", 0.55)]:
    bars = ax.barh(y + off, h[col], w * 0.92, color=[COLOR[n] for n in h.index], alpha=shade)
    ax.bar_label(bars, labels=[f"{v:.2f}" for v in h[col]], fontsize=7, color=INK2, padding=2)
ax.set_yticks(y, h.index); ax.invert_yaxis(); ax.set_xlim(0, 1.05)
ax.set_title(f"Top-1 (dark) and decision accuracy (light), {SHOWN}"); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

**Reading note.** The decision accuracy of `random` is about the no_match share (about 0.20): the threshold chooses "always abstain" because there is no signal. A baseline's value is how far
above this floor and above the word-overlap baselines it sits. Top-3 is already 1.0 for examples with k <= 3 (about 40%), so it must be compared with `random`.

## 2. Top-1 per dataset

In [ ]:
datasets = sorted(next(iter(RES.values()))["splits"][SHOWN]["datasets"])
by_ds = pd.DataFrame({n: {ds: r["splits"][SHOWN]["datasets"][ds]["top1"] for ds in datasets} for n, r in RES.items()}).T
display(by_ds.round(3))
dec_ds = pd.DataFrame({n: {ds: r["splits"][SHOWN]["datasets"][ds]["decision_accuracy"] for ds in datasets} for n, r in RES.items()}).T
print("Decision accuracy (global τ):")
display(dec_ds.round(3))
own = pd.DataFrame({n: {ds: r["splits"][SHOWN]["datasets_own_threshold"][ds]["decision_accuracy"] for ds in datasets} for n, r in RES.items()}).T
print("Decision accuracy (separate τ per dataset; in real use every deployment calibrates on its own data):")
display(own.round(3))

## 3. Generalisation: seen vs. unseen conditions

The project's central question. Three groups:
* **exact**: the same condition was seen as gold in training,
* **equivalent_only**: a *different* condition with the same meaning (another service or another wording) was seen,
* **unseen**: not even an equivalent exists in training.

For training-free baselines "seen" gives no advantage (nothing is learned); this table still shows **where models trained later start from**, what they have to prove, and how hard the slices are by nature.

In [ ]:
SLICES = [("seen_exact:exact", "seen\n(exact)"), ("seen_exact:equivalent_only", "seen\n(equivalent only)"), ("cond:unseen", "unseen")]
sl = pd.DataFrame({n: {lab.replace("\n", " "): r["splits"][SHOWN]["slices"].get(k, {}).get("top1") for k, lab in SLICES} for n, r in RES.items()}).T
n_sl = {lab.replace("\n", " "): next(iter(RES.values()))["splits"][SHOWN]["slices"].get(k, {}).get("n") for k, lab in SLICES}
print("example counts (route):", n_sl)
display(sl.round(3))

fig, ax = plt.subplots(figsize=(10, 3.8))
names = list(RES); x = np.arange(len(SLICES)); w = 0.8 / len(names)
for i, n in enumerate(names):
    vals = [RES[n]["splits"][SHOWN]["slices"].get(k, {}).get("top1") or 0 for k, _ in SLICES]
    bars = ax.bar(x + (i - (len(names) - 1) / 2) * w, vals, w * 0.9, color=COLOR[n], label=n)
    ax.bar_label(bars, labels=[f"{v:.2f}" for v in vals], fontsize=6.5, color=INK2, padding=1.5, rotation=90)
ax.set_xticks(x, [lab for _, lab in SLICES]); ax.set_ylim(0, 1.12); ax.set_ylabel("Top-1")
ax.set_title(f"Top-1: seen and unseen conditions ({SHOWN})"); ax.legend(ncol=4, fontsize=8, loc="upper left"); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

In [ ]:
OTHER = [g for g in ["ctx:none", "ctx:first_turn", "ctx:later_turn", "svc:seen", "svc:unseen", "k:2-3", "k:4-6", "k:7+"]
         if g in next(iter(RES.values()))["splits"][SHOWN]["slices"]]
print("Top-1, other slices:")
display(pd.DataFrame({n: {g: r["splits"][SHOWN]["slices"][g]["top1"] for g in OTHER} for n, r in RES.items()}).T.round(3))
print("Decision accuracy, other slices (global τ):")
display(pd.DataFrame({n: {g: r["splits"][SHOWN]["slices"][g]["decision_accuracy"] for g in OTHER} for n, r in RES.items()}).T.round(3))

## 4. Paraphrase robustness (SGD-X twins)

In [ ]:
rows = {}
for n, r in RES.items():
    pc = r["splits"][SHOWN]["paraphrase_consistency"]
    rows[n] = {"pairs": pc["n_pairs"], "Top-1 original": pc["top1_main"], "Top-1 paraphrase": pc["top1_paraphrase"],
               "same pick": pc["same_selected_candidate"], "same abstain decision": pc["same_abstain_decision"]}
pdf = pd.DataFrame(rows).T
display(pdf.round(3))

fig, ax = plt.subplots(figsize=(8.5, 3.4))
y = np.arange(len(pdf))
for i, n in enumerate(pdf.index):
    a, b = pdf.loc[n, "Top-1 original"], pdf.loc[n, "Top-1 paraphrase"]
    ax.plot([a, b], [i, i], color="#c9c8c2", lw=2, zorder=1)
    ax.scatter([a], [i], s=36, color=COLOR[n], zorder=2)
    ax.scatter([b], [i], s=36, facecolor="white", edgecolor=COLOR[n], lw=1.8, zorder=2)
    ax.text(max(a, b) + 0.015, i, f"{a:.2f} → {b:.2f}", va="center", fontsize=7.5, color=INK2)
ax.set_yticks(y, pdf.index); ax.invert_yaxis(); ax.set_xlim(0, 1.0); ax.set_xlabel("Top-1")
ax.set_title(f"SGD-X: original (filled) -> held-out paraphrase (hollow), {SHOWN}"); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

## 5. no_match behaviour

In [ ]:
NM = ["no_match_precision", "no_match_recall", "no_match_f1", "false_routing_rate", "abstain_rate", "false_abstain_rate", "wrong_route_rate"]
print(f"== {SHOWN}: global τ ==")
display(pd.DataFrame({n: {k: r["splits"][SHOWN]["overall"][k] for k in NM} for n, r in RES.items()}).T.round(3))
kinds = [g for g in ["nm_kind:other_family", "nm_kind:sibling_removed", "nm_kind:oos"] if g in next(iter(RES.values()))["splits"][SHOWN]["slices"]]
print("Abstention by no_match kind (recall):")
display(pd.DataFrame({n: {g: r["splits"][SHOWN]["slices"][g]["no_match_recall"] for g in kinds} for n, r in RES.items()}).T.round(3))
print("τ values:", {n: round(r["threshold"]["global"], 3) for n, r in RES.items()})

## 6. Threshold curve (validation only)

Recomputes the scores of the best baseline on validation and shows the trade-off between correct routing, false routing and abstention as τ varies.
It is a first look at a risk-sensitive policy: how much correct routing does it cost to reduce false routing?

In [ ]:
from pathwiser.evaluation.metrics import predictions, summarize
from pathwiser.scorers.registry import build
from pathwiser.storage.dataset import load_split

best = max((n for n in RES if n != "random"), key=lambda n: RES[n]["splits"]["val"]["overall"]["decision_accuracy"])
print("best baseline (validation decision accuracy):", best)
val = load_split("val", ROOT / "data" / "model" / "v1")
scorer = build(best)
scorer.fit([])
scores = scorer.score(val)
preds = [p for p in predictions(val, scores) if not p.paired]
tops = np.array([p.top for p in preds])
grid = np.quantile(tops, np.linspace(0.0, 1.0, 101))
curve = pd.DataFrame([{"tau": t, **{k: v for k, v in summarize(preds, float(t)).items()
                                    if k in ("decision_accuracy", "route_accuracy", "false_routing_rate", "abstain_rate",
                                             "no_match_precision", "no_match_recall")}} for t in grid])

fig, ax = plt.subplots(figsize=(8.5, 3.8))
for col, lab, c in [("route_accuracy", "correct routing (on route examples)", PAL[2]),
                    ("false_routing_rate", "false routing (on all examples)", PAL[7]),
                    ("no_match_recall", "no_match detection (recall)", PAL[0])]:
    ax.plot(curve["tau"], curve[col], color=c, lw=2, label=lab)
tau_star = RES[best]["threshold"]["global"]
ax.axvline(tau_star, color=INK2, ls="--", lw=1)
ax.text(tau_star, 1.02, f" tuned τ = {tau_star:.2f}", fontsize=8, color=INK2, va="bottom")
ax.set_xlabel("τ (abstain if the best score is below it)"); ax.set_ylim(0, 1.08); ax.legend(loc="center left", fontsize=8)
ax.set_title(f"Trade-off as the threshold varies: {best}, validation")
plt.tight_layout(); plt.show()

## 7. Findings (test, τ from validation)

1. **Pretrained sentence embeddings are clearly ahead.** `st_message` Top-1 0.846 / decision accuracy 0.725; the best word-based baseline (`lexical`) 0.616 / 0.579; chance 0.242 / 0.227.
2. **Adding the context as is does not help and can hurt.** `st_query` (node + context + message) drops to 0.621 (`st_message` 0.846); `lexical_ctx` and `tfidf_ctx` are not meaningfully better than their context-free versions either.
   Using the context takes a trained model (cross-encoder / fine-tuning).
3. **No seen/unseen gap, which is expected.** For a training-free model "seen" is no advantage (`st_message`: exact 0.871, equivalent-only 0.808, unseen 0.845).
   The slices differ in difficulty, so these numbers are **not evidence of generalisation**; the evidence is a trained model beating this reference on the same slices.
4. **Weak spots.**
   * ABCD collapses on test (`st_message` Top-1 0.643; 0.841 on validation). The `ctx:first_turn` slice (ABCD only) is weak for that reason. The held-out `storewide_query` family and the long, similar ABCD descriptions are suspects.
   * On SGD-X held-out paraphrases `st_message` goes from 0.809 to 0.758; the share of picking the same candidate is only 0.691. Sensitivity to description wording is real.
5. **no_match is weak with a single threshold.** Best no_match precision/recall 0.587/0.642. Abstention on `sibling_removed` is only 0.459 (`other_family` 0.652, `oos` 0.923): when the close sibling of the right condition stays among the candidates, the best score stays high.
   This is the main motivation for the decision-layer work (margins, calibration, risk-sensitive policies; see `docs/decision-layer.md`).

**Limits.** These baselines learn nothing; they are the reference the next step has to beat. High word overlap shows that the data is partly solvable superficially.
Most condition descriptions (except SGD's) are unreviewed LLM drafts. Latency numbers are batch measurements on a CPU, a rough indicator.